# Implementing GPT model from scratch to generate text

In [47]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # vocabulary size
    "context_length": 1024, # context length
    "emb_dim": 768, # embedding dimension
    "n_heads": 12,  # no. of attention heads
    "n_layers": 12, # no. of transformer layers
    "drop_rate": 0.1, # dropout rate
    "qkv_bias": False # query-key-value bias
}

# Feed forward with GeLU activation

Advantages of GeLU over ReLU.
1. Differentiable. Each value of x corresponds to its different value.(not the same 0 for negatives)
2. Solves dead neurons problem, as negative values will also contribute to training as it is not 0.
3. As per experiments, GeLU provided great results compared to other activation functions in GPTs.

In [48]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2 / torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
    ))

In [49]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), # Expansion
        GELU(), # Activation
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), # Contraction
    )

  def forward(self, x):
    return self.layers(x)

In [50]:
print(GPT_CONFIG_124M["emb_dim"])

768


In [51]:
ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2,3,768)
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


## MultiHead Attention

In [52]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()

    assert (d_out % num_heads) == 0, "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    # calculate individual head dimension according to d_out and no. of heads present
    self.head_dim = d_out // num_heads

    # random key,query,value initialization with d_in and d_out)
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.out_proj = nn.Linear(d_in, d_out) # Linear layer to combine head outputs
    self.dropout = nn.Dropout(dropout)

    self.register_buffer(
        "mask",
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    b, num_tokens, d_in = x.shape # initialize (Batch, token_size, input_dimension)

    # keys, values queries (random of d_in,d_out(dimensions) multiplied with inputs)
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    # convert of each head i.e d_out --> num_heads and head_dimension
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Group matrices by num_heads for parallel computation.

    #(b,num_tokens,num_heads,head_dim) --> (b, num_heads, num_tokens, head_dim)
    # (1,3,2,3) --> (1,2,3,3) (The positions 1 and 2 will be transposed)
    keys = keys.transpose(1,2)
    queries = queries.transpose(1,2)
    values = values.transpose(1,2)

    # now for each query we will do matmul with keys.
    # and for that we need to transpose the postion 2 and 3 of keys.
    # (b,num_heads,num_tokens,head_dim) * (b, num_heads, head_dim, num_tokens)
    #                                    |
    #                    (b,num_heads,num_tokens,num_tokens)
    attn_scores = queries @ keys.transpose(2,3)

    # masking
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)

    # softmax with Sqrt of head_dim and dropout
    attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # calulate context vector with d_out as dimension preserved

    # (b,num_heads,num_tokens,num_tokens) * (b,num_heads,num_tokens,head_dim)
    #                                     |
    #                     (b,num_heads,num_tokens,head_dim)
    #                                     | (1,2) transpose
    #                     (b,num_tokens,num_heads,head_dim)
    context_vector = (attn_weights @ values).transpose(1,2)
    # now we can merge num_heads and head_dim easily to d_out.
    # we merge the num_heads and head_dim into single row giving d_out dimension.
    # (b,num_tokens,num_heads,head_dim) --> (b,num_tokens,d_out)
    # contiguous ensures that after reshaping the values stay in same block of memory.
    context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out)
    context_vector = self.out_proj(context_vector)

    return context_vector

# **GPT architecture**

In [53]:
import torch
import torch.nn as nn

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    # Use a placeholder for transformer block
    self.trf_blocks = nn.Sequential(
        *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    # Use a placeholder for layer norm
    self.final_norm = LayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att = MultiHeadAttention(
        d_in = cfg["emb_dim"],
        d_out = cfg["emb_dim"],
        context_length = cfg["context_length"],
        num_heads = cfg["n_heads"],
        dropout = cfg["drop_rate"],
        qkv_bias = cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg)
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # shortcut connection for attention block
    shortcut = x
    x = self.norm1(x) # normalization
    x = self.att(x) # attention
    x = self.drop_shortcut(x) # dropout
    x = x + shortcut # add the original input back

    # shortcut connection for feed forward block
    shortcut = x
    x = self.norm2(x) # normalization
    x = self.ff(x) # feed forward
    x = self.drop_shortcut(x) # dropout
    x = x + shortcut # add the original input back

    return x

class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    # if unbiased is 'True', it applied Bessels correction which is divide by n-1 for variance not by n.
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.eps) # eps --> Epsilon is used to prevent division by 0 during normalization.
    return self.scale * norm_x + self.shift # scale and shifts are trainable parameters used to tweak norms.

In [54]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [55]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
logits = model(batch)
# each token will have probabilistic value for every vocab size and we will pick with highest one
# so the output shape will be like 2 batch size, 4 input tokens for each batch and each token with value for 50257 vocabs.
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


Printing total no. of parameters

In [57]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


In [58]:
print("Token embedding layer shape: ", model.tok_emb.weight.shape)
print("Output head layer shape: ", model.out_head.weight.shape)

Token embedding layer shape:  torch.Size([50257, 768])
Output head layer shape:  torch.Size([50257, 768])


In GPT-2, the same token embedding parameter is used as output head parameters giving it less no. of parameters which is 124 million.

In [60]:
total_params_GPT2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Total number of parameters: {total_params_GPT2:,}")

Total number of parameters: 124,412,160


Space taken by our model with 163million parameters

In [66]:
total_size_bytes = total_params * 4
total_size_mb = total_size_bytes / (1024*1024)
print(f"Total size of the model: {total_size_mb:.2f}mb")

Total size of the model: 621.83mb
